# RDFLIB

On peut utiliser les librairies de https://rdflib.dev
Par exemple SPARQLWrapper facilite le lancement de requêtes sparql. 
On utilisera pandas de manière classique pour traiter les résultats de requêtes comme des tableaux.



In [7]:
!pip install SPARQLWrapper pandas networkx pyvis

#SPARQLWrapper

In [8]:
from SPARQLWrapper import SPARQLWrapper, JSON
import pandas as pd

def query_dbpedia(movie_name):
    
    sparql = SPARQLWrapper("https://dbpedia.org/sparql")
    
    # Requête complexe : Albums, dates et genres
    query = f"""
    PREFIX dbo: <http://dbpedia.org/ontology/>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

    SELECT DISTINCT ?director ?release_date ?genre
    WHERE {{
      ?film rdf:type dbo:Film;
            dc:title "{movie_name}" .
      ?film dbo:director ?director ;
             dbo:releaseDate ?release_date .
      OPTIONAL {{ 
        ?film dbo:genre ?genre .
      }}
    }}
    ORDER BY DESC(?release_date)
    LIMIT 50
    """
    
    sparql.setQuery(query)
    sparql.setReturnFormat(JSON)
    results = sparql.query().convert()
    
    # Requête et transformation du résultats en pandas dataframe
    data = []
    for result in results["results"]["bindings"]:
        data.append({
            "Film": movie_name,
            "Realisateur" : result["director"]["value"],
            "Date": result.get("release_date", {}).get("value", "N/A"),
            "Genre": result.get("genre_label", {}).get("value", "N/A")
        })
    return pd.DataFrame(data)

# Lancement de la requête
df_music = query_dbpedia("Michael Jackson")
df_music.head()

""


# Exploration DBPedia avec SPARQLWrapper

# Exploration de HAL 

Quelques éléments de modélisation : https://fr.slideshare.net/slideshow/dcouverte-du-sparql-endpoint-de-hal/120942343#2

In [9]:
def query_coauthors_by_author(target_author_name):
    """
    Récupère la liste des co-auteurs et des titres de publications pour un auteur donné sur HAL.
    """
    endpoint = "https://data.archives-ouvertes.fr/sparql"
    sparql = SPARQLWrapper(endpoint)
    
    # On utilise des f-strings pour injecter l'auteur
    # Note : On ajoute un FILTER pour ne pas s'inclure soi-même dans la liste des co-auteurs
    query = f"""
    PREFIX foaf: <http://xmlns.com/foaf/0.1/>
    PREFIX dc: <http://purl.org/dc/terms/>

    SELECT DISTINCT ?coauthor_name ?title
    WHERE {{
      ?pub dc:title ?title .
      ?pub dc:creator ?a1 .
      ?pub dc:creator ?a2 .
      
      # L'auteur cible doit être l'un des créateurs
      ?a1 foaf:name "{target_author_name}" .
      
      # On récupère le nom de l'autre créateur (le co-auteur)
      ?a2 foaf:name ?coauthor_name .
      
      # On évite de se lister soi-même comme co-auteur
      FILTER (str(?coauthor_name) != "{target_author_name}")
    }}
    ORDER BY ?coauthor_name
    LIMIT 100
    """
    
    sparql.setQuery(query)
    sparql.setReturnFormat(JSON)
    results = sparql.query().convert()
    
    # Extraction vers un DataFrame
    data = []
    for r in results["results"]["bindings"]:
        data.append({
            "Source": target_author_name,
            "Target": r["coauthor_name"]["value"],
            "Publication": r["title"]["value"]
        })
    
    return pd.DataFrame(data)

# --- Test de la fonction ---
df_mehdi = query_coauthors_by_author("Mehdi Kaytoue")

print(f"Nombre de relations de co-autorat trouvées : {len(df_mehdi)}")
df_mehdi.head(10)

Nombre de relations de co-autorat trouvées : 6


,Source,Target,Publication
0,Mehdi Kaytoue,Amedeo Napoli,Three Views on Dependency Covers from an FCA P...
1,Mehdi Kaytoue,Amedeo Napoli,Dependency Covers from a FCA Perspective
2,Mehdi Kaytoue,Jaume Baixeries,Three Views on Dependency Covers from an FCA P...
3,Mehdi Kaytoue,Jaume Baixeries,Dependency Covers from a FCA Perspective
4,Mehdi Kaytoue,Victor Codocedo,Three Views on Dependency Covers from an FCA P...
5,Mehdi Kaytoue,Victor Codocedo,Dependency Covers from a FCA Perspective


# Graphe de co-auteurs et visualisation sous forme de graphe

In [10]:
from SPARQLWrapper import SPARQLWrapper, JSON
import pandas as pd

def get_weighted_coauthors(target_name):
    endpoint = "https://data.archives-ouvertes.fr/sparql"
    sparql = SPARQLWrapper(endpoint)
    
    query = f"""
    PREFIX dcterms: <http://purl.org/dc/terms/>
    PREFIX foaf: <http://xmlns.com/foaf/0.1/>
    PREFIX hal: <http://data.archives-ouvertes.fr/schema/>
    
    SELECT ?co_nom (COUNT(?version) AS ?poids)
    WHERE {{
      ?target_auteur a foaf:Person ; foaf:name "{target_name}" .
      ?target_u hal:person ?target_auteur .
      ?version dcterms:creator ?target_u .
      ?version dcterms:creator ?co_u .
      ?co_u hal:person ?co_auteur .
      ?co_auteur foaf:name ?co_nom .
    }}
    GROUP BY ?co_nom
    ORDER BY DESC(?poids) 
    """
    
    sparql.setQuery(query)
    sparql.setReturnFormat(JSON)
    results = sparql.query().convert()
    
    data = []
    for r in results["results"]["bindings"]:
        data.append({
            "Source": target_name,
            "Target": r["co_nom"]["value"],
            "Weight": int(r["poids"]["value"])
        })
    return pd.DataFrame(data)

# Extraction des données
df_weighted = get_weighted_coauthors("Mehdi Kaytoue")

print(df_weighted.head())

          Source                   Target  Weight
0  Mehdi Kaytoue            Mehdi Kaytoue     112
1  Mehdi Kaytoue            Amedeo Napoli      50
2  Mehdi Kaytoue  Jean-François Boulicaut      26
3  Mehdi Kaytoue           Marc Plantevit      18
4  Mehdi Kaytoue          Victor Codocedo      18


In [11]:
from pyvis.network import Network

def visualize_ego_network(df, target_name):
    # Initialisation du graphe interactif
    net = Network(height="600px", width="100%", bgcolor="#222222", font_color="white", notebook=True)
    
    # Ajout du nœud central (l'auteur)
    net.add_node(target_name, label=target_name, color="#ff4d4d", size=30)
    
    # Ajout des co-auteurs et des arêtes pondérées
    for _, row in df.iterrows():
        # La taille du nœud et l'épaisseur du lien dépendent du poids (nombre de publications)
        net.add_node(row['Target'], label=row['Target'], color="#4d94ff", size=10 + row['Weight']*2)
        net.add_edge(target_name, row['Target'], value=row['Weight'], title=f"{row['Weight']} publications")
    
    # Configuration de la physique pour un rendu esthétique
    net.force_atlas_2based()
    return net.show("coauthor_network.html")

# Affichage du graphe
visualize_ego_network(df_weighted, "Mehdi Kaytoue")

coauthor_network.html


# HAL Graphe des co-auteurs de niveau 2

On peut écrire le graphe dans un CSV et l'étudier avec des logiciels tiers, algorithmes d'analyse etc.

In [12]:
from SPARQLWrapper import SPARQLWrapper, JSON
import pandas as pd
import time

def get_coauthors_recursive(target_name, depth=1, visited=None):
    if visited is None:
        visited = set()
    
    # Éviter de traiter deux fois la même personne ou de tourner en rond
    if target_name in visited or depth < 0:
        return pd.DataFrame()
    
    visited.add(target_name)

     # ici il faut un endpoint qui tient la route. Le endpoint de HAL est lent. Celui que j'ai installé localement très rapide (quelques secondes)
     # Celui du liris mieux que HAL, mais très lent. 
    
    #endpoint = "http://localhost:7200/repositories/hal-archives"
    #endpoint = "https://data.archives-ouvertes.fr/sparql"
    endpoint = "https://graphdb.pagoda.liris.cnrs.fr/repositories/hal" 
    
    sparql = SPARQLWrapper(endpoint)
    
    # Requête pour trouver les co-auteurs directs
    query = f"""
    PREFIX dcterms: <http://purl.org/dc/terms/>
    PREFIX foaf: <http://xmlns.com/foaf/0.1/>
    PREFIX hal: <http://data.archives-ouvertes.fr/schema/>
    
    SELECT DISTINCT ?co_nom (COUNT(?version) AS ?poids)
    WHERE {{
      ?target_auteur a foaf:Person ; foaf:name "{target_name}" .
      ?target_u hal:person ?target_auteur .
      ?version dcterms:creator ?target_u .
      ?version dcterms:creator ?co_u .
      ?co_u hal:person ?co_auteur .
      ?co_auteur foaf:name ?co_nom .
    }}
    GROUP BY ?co_nom
    HAVING (COUNT(?version) >= 10)
    """
    
    sparql.setQuery(query)
    sparql.setReturnFormat(JSON)
    
    try:
        results = sparql.query().convert()
    except Exception as e:
        print(f"Erreur pour {target_name}: {e}")
        return pd.DataFrame()

    all_edges = []
    current_level_authors = []

    for r in results["results"]["bindings"]:
        co_nom = r["co_nom"]["value"]
        weight = int(r["poids"]["value"])
        all_edges.append({"Source": target_name, "Target": co_nom, "Weight": weight})
        current_level_authors.append(co_nom)

    df_current = pd.DataFrame(all_edges)

    # Si on doit descendre plus bas (niveau 2)
    if depth > 0:
        for author in current_level_authors:
            # Petite pause pour respecter le serveur HAL
            print (author)
            time.sleep(0.1) 
            df_next = get_coauthors_recursive(author, depth - 1, visited)
            df_current = pd.concat([df_current, df_next], ignore_index=True)

    return df_current

# --- Exécution ---
# Attention : depth=1 récupère vos co-auteurs ET leurs co-auteurs.
# Cela peut prendre 1 à 2 minutes selon le nombre de collaborateurs.
df_extended = get_coauthors_recursive("Mehdi Kaytoue", depth=1)

# Nettoyage final : on enlève les lignes vides et on peut sauvegarder en CSV
df_extended = df_extended.drop_duplicates()
print(f"Nombre de relations trouvées : {len(df_extended)}")
print(df_extended)

Zainab Assaghir
Mehdi Kaytoue
Amedeo Napoli
Youcef Remil
Romain Mathonat
Jean-François Boulicaut
Marc Plantevit
C. Robardet
Victor Codocedo
Chedy Raïssi
Guillaume Bosc
Nombre de relations trouvées : 94
            Source                   Target  Weight
0    Mehdi Kaytoue          Zainab Assaghir      14
1    Mehdi Kaytoue            Mehdi Kaytoue     112
2    Mehdi Kaytoue            Amedeo Napoli      50
3    Mehdi Kaytoue             Youcef Remil      13
4    Mehdi Kaytoue          Romain Mathonat      11
..             ...                      ...     ...
89    Chedy Raïssi               Elias Egho      13
90    Chedy Raïssi              Nicolas Jay      15
91  Guillaume Bosc           Guillaume Bosc      12
92  Guillaume Bosc            Mehdi Kaytoue      12
93  Guillaume Bosc  Jean-François Boulicaut      11

[94 rows x 3 columns]
